In [ ]:
from PIL import Image, ImageOps
import numpy as np
import cv2
import os
import cupy as cp
import matplotlib.pyplot as plt
from ultralytics import YOLO
import glob

In [ ]:
# bboxの学習
model = YOLO('yolov8l.pt')

file = "hukusuu_train"

results = model.train(
    data=f'/home/YOLO/{file}/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project=f"/home/YOLO/{file}/datasets", 
    device='mps'   
)

In [ ]:
# segmantationの学習
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='/home/YOLO/0930_mask/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project="/home/YOLO/0930_mask/datasets", 
    device='mps'   
)

In [ ]:
model = YOLO('/home/YOLO/datasets/train4/weights/best.pt')

# Predict the model
# model.predict('/home/data/0410/A/Atest/IMG_1364.JPEG', save = False,save_crop=True, retina_masks = True, conf=0.5)
results = model.predict('/home/data/0410/A/Atest/IMG_1367.JPEG')
for i, r in enumerate(results):
    # Plot results image
    im_bgr = r.plot(masks=True)  # BGR-order numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image

    # Show results to screen (in supported environments)
    # r.show()
    r.show(masks=True,boxes=False)

    # Save results to disk
    # r.save(filename=f"results{i}.jpg")

cropされるとき少しパディングされるので、パディングなしでcropしたい場合は、githubからリポジトリをcloneし、plotting.pyのsave_one_boxのパディングの値を0に変更してから実行する必要がある。→したけど変わらん

resultsにすべて入っているっぽいのでそこからどうにかして一番信頼度高いのとってくる


In [ ]:


# モデルの初期化
model = YOLO('/home/YOLO/datasets/train4/weights/best.pt')

# 予測を行いたいフォルダ
folders = ["/home/data/shiitake_maybeuse/A",
        #    "/home/data_org/B/B",
        #    "/home/data_org/CD/CD"
           ]

# 各フォルダに対してループ
for folder in folders:
    # フォルダ内の全JPEGファイルを取得
    files = glob.glob(f"{folder}/*.JPEG")
    # 各ファイルに対して予測を実行
    for file_path in files:
        filename = os.path.basename(file_path)
        model.predict(file_path,  save=True, retina_masks=False, conf=0.5,name=f"/home/data/yolo_seg/A/{filename}")

In [ ]:
#save mask shiitake 

# model = YOLO('/home/YOLO/mask/datasets/train/weights/best.pt')
model = YOLO('/home/YOLO/0930_mask/datasets/train/weights/best.pt')


folders = [
        #   "/home/data/yolo_crop/A",
        #   "/home/data/yolo_crop/B",
        #   "/home/data/yolo_crop/CD",
          "/home/data/0930/crop"
          ]
for folder in folders:
    files = glob.glob(f"{folder}/*.jpg")
    for file_path in files:
        name = os.path.basename(file_path)
        name = name.replace(".jpg","")
         # 画像を読み込んでサイズを取得
        image = cv2.imread(file_path)
        height, width = image.shape[:2]

        # 読み込んだ画像と同じ大きさの黒い画像を作成
        black_image = np.zeros((height, width), dtype=np.uint8)

        results = model(file_path)  # results list
        if folder == "/home/data/yolo_crop/A":
            for i, r in enumerate(results):
                max_idx = np.argmax(results[0].boxes.conf)
                points = np.array(r.masks.xy[max_idx], dtype=np.int32)
                cv2.fillPoly(black_image, [points], 255)
                cv2.imwrite(f"/home/data/yolo_mask_crop/A/{name}.jpg", black_image)
        elif folder == "/home/data/yolo_crop/B":
            for i, r in enumerate(results):
                max_idx = np.argmax(results[0].boxes.conf)
                points = np.array(r.masks.xy[max_idx], dtype=np.int32)
                cv2.fillPoly(black_image, [points], 255)
                cv2.imwrite(f"/home/data/yolo_mask_crop/B/{name}.jpg", black_image)
        elif folder == "/home/data/yolo_crop/CD":
            for i, r in enumerate(results):
                max_idx = np.argmax(results[0].boxes.conf)
                points = np.array(r.masks.xy[max_idx], dtype=np.int32)
                cv2.fillPoly(black_image, [points], 255)
                cv2.imwrite(f"/home/data/yolo_mask_crop/CD/{name}.jpg", black_image)
        elif folder == "/home/data/0930/crop":
            for i, r in enumerate(results):
                max_idx = np.argmax(results[0].boxes.conf)
                points = np.array(r.masks.xy[max_idx], dtype=np.int32)
                cv2.fillPoly(black_image, [points], 255)
                cv2.imwrite(f"/home/data/0930/mask/{name}.jpg", black_image)

In [ ]:
#crop

# YOLOモデルのロード
# model = YOLO('/home/YOLO/bbox2/datasets/train/weights/best.pt')
model = YOLO('/home/YOLO/1008_bbox/datasets/train/weights/best.pt')

date = "1216"
# 処理するフォルダのリスト
folders = [
    f"/home/data/{date}/A",
    f"/home/data/{date}/B",
    f"/home/data/{date}/C",
]

for folder in folders:
    files = glob.glob(f"{folder}/*.JPEG")
    for file_path in files:
        name = os.path.basename(file_path).replace(".JPEG", "")

        # 推論を実行
        results = model(file_path, conf=0.1)  # results list
        if results and len(results[0].boxes.conf) > 0:  # confが空でないことを確認
            for i, r in enumerate(results):
                max_idx = np.argmax(results[0].boxes.conf)
                start_x, start_y, end_x, end_y = map(int, results[0].boxes.xyxy[max_idx])
                clip_img = results[0].orig_img[start_y:end_y, start_x:end_x]

                # 画像を保存
                if folder == f"/home/data/{date}/A":
                    save_path = f"/home/data/{date}/crop/A/{name}.jpg"
                elif folder == f"/home/data/{date}/B":
                    save_path = f"/home/data/{date}/crop/B/{name}.jpg"
                elif folder == f"/home/data/{date}/CD":
                    save_path = f"/home/data/{date}/crop/C/{name}.jpg"
                elif folder == f"/home/data/{date}":
                    save_path = f"/home/data/{date}/crop/{name}.jpg"
                cv2.imwrite(save_path, clip_img)

In [ ]:
#save crop shiitake test


model = YOLO('/home/YOLO/bbox2/datasets/train/weights/best.pt')
# model = YOLO('/home/YOLO/0917_bbox/datasets/train/weights/best.pt')

path = "/home/data/shiitake_maybeuse/B/IMG_1277.JPEG"
results = model(path)  # results list
name = "1266"
#save_crop:save_dir/cls/file_nameに保存
# Visualize the results
for i, r in enumerate(results):
    
    print("確率:conf")
    print(results[0].boxes.conf)
    print("最大のindex")
    print(np.argmax(results[0].boxes.conf))
    max_idx = np.argmax(results[0].boxes.conf)
    print("bbox:xywh")
    print(results[0].boxes.xyxy)
    print("最大のbbox:xywh")
    print(results[0].boxes.xyxy[max_idx])

    print(results[0].boxes)

    cv2.imwrite("orig_img.jpg", results[0].orig_img)
    start_x, start_y, end_x, end_y = map(int, results[0].boxes.xyxy[max_idx])
    clip_img = results[0].orig_img[start_y:end_y,start_x:end_x]
    cv2.imwrite("clip_img.jpg", clip_img)

    r.show()

In [ ]:
import numpy as np
import cv2

# 640x480の黒画像を作成
black_image = np.zeros((1536, 2048), dtype=np.uint8)

# YOLOモデルの読み込みと推論
# model = YOLO('/home/YOLO/mask/datasets/train/weights/best.pt')
model = YOLO('/home/YOLO/0917_bbox/datasets/train/weights/best.pt')
# model = YOLO('/home/YOLO/bbox2/datasets/train/weights/best.pt')
path = "/home/data/0930/IMG_1620.JPEG"
results = model(path)  # results list

# 結果のマスク座標を取得し、黒画像に白で描写
for i, r in enumerate(results):
    # print(f"Result {i}:")
    # for mask in r.masks.xy:
    #     print(mask)  # マスク座標をプリント
    #     # 座標を整数に変換
    #     points = np.array(mask, dtype=np.int32)
    #     # ポリゴンを白で描写
    #     cv2.fillPoly(black_image, [points], 255)
    
    r.show(boxes=True,masks=True)

# 結果を保存
# cv2.imwrite("masked_image.jpg", black_image)

In [ ]:
import os
import cv2
from pathlib import Path
from ultralytics import YOLO

# --- 設定項目 ---
# 1. モデルのパス
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'

# 2. 画像が保存されているフォルダのパス
IMAGE_DIR = Path('/home/data/0707_sam/data')

# 3. クロップ画像を保存するフォルダのパス
SAVE_DIR = IMAGE_DIR / 'crop'

# 4. 切り取り範囲の余白 (割合で指定: 0.1 = 10%の余白を追加)
PADDING_RATIO = 0.03
# -----------------

# 保存先フォルダを作成 (存在しない場合)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"クロップ画像の保存先: {SAVE_DIR}")

# モデルを読み込む
model = YOLO(MODEL_PATH)

# 画像フォルダ内の画像ファイルに対して処理を実行
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
for image_path in IMAGE_DIR.iterdir():
    if image_path.suffix.lower() in image_extensions:
        print(f"処理中の画像: {image_path.name}")

        try:
            # 元の画像を読み込む
            img = cv2.imread(str(image_path))
            if img is None:
                print(f"  -> スキップ: 画像を読み込めませんでした。")
                continue

            # 元画像の高さと幅を取得
            img_height, img_width, _ = img.shape

            # モデルで推論を実行
            results = model.predict(source=img, verbose=False)

            # 検出された各オブジェクトに対して処理
            for i, result in enumerate(results):
                boxes = result.boxes
                for j, box in enumerate(boxes):
                    # 座標を取得 (x1, y1, x2, y2)
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # ⬇️ここからが変更点⬇️

                    # 1. 余裕を持たせるためのパディング量を計算
                    box_width = x2 - x1
                    box_height = y2 - y1
                    padding_w = int(box_width * PADDING_RATIO)
                    padding_h = int(box_height * PADDING_RATIO)

                    # 2. パディングを適用した新しい座標を計算
                    x1_pad = x1 - padding_w
                    y1_pad = y1 - padding_h
                    x2_pad = x2 + padding_w
                    y2_pad = y2 + padding_h

                    # 3. 座標が画像の範囲外に出ないようにクリッピング
                    x1_final = max(0, x1_pad)
                    y1_final = max(0, y1_pad)
                    x2_final = min(img_width, x2_pad)
                    y2_final = min(img_height, y2_pad)

                    # ⬆️ここまでが変更点⬆️

                    # クラス名を取得
                    class_id = int(box.cls[0])
                    class_name = model.names[class_id]

                    # パディング適用後の座標で画像をクロップ
                    cropped_img = img[y1_final:y2_final, x1_final:x2_final]

                    # 保存ファイル名を生成
                    original_stem = image_path.stem
                    save_filename = f"{original_stem}_{class_name}_{j}.jpg"
                    save_path = SAVE_DIR / save_filename

                    # クロップ画像を保存 (念のため、クロップ結果が空でないかチェック)
                    if cropped_img.size > 0:
                        cv2.imwrite(str(save_path), cropped_img)
                    else:
                        print(f"  -> スキップ: クロップ後の画像サイズが0になりました。")

        except Exception as e:
            print(f"  -> エラーが発生しました: {e}")

print("\nすべての処理が完了しました。")

In [ ]:
#0728_segmetのYOLO,0707_samのデータセット
#cropの画像からマスク作成
import os
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

# --- 設定 ---
# 1. 学習済みモデルのパス
MODEL_PATH = '/home/YOLO/0728_segmet/datasets/train2/weights/best.pt'

# 2. 入力画像のフォルダパス
INPUT_DIR = '/home/data/0707_sam/data/crop'

# 3. マスク画像の保存先フォルダパス
OUTPUT_DIR = '/home/data/0707_sam/data/mask'
# ---

def create_and_save_masks():
    """
    指定されたフォルダ内の全画像に対してセグメンテーションを実行し、
    検出されたオブジェクトのマスクを結合して保存する。
    """
    # YOLOモデルを読み込む
    try:
        model = YOLO(MODEL_PATH)
        print(f"モデル '{MODEL_PATH}' を正常に読み込みました。")
    except Exception as e:
        print(f"エラー: モデルの読み込みに失敗しました。パスを確認してください。: {e}")
        return

    # 保存先フォルダが存在しない場合は作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"マスクの保存先: '{OUTPUT_DIR}'")

    # 処理対象の画像ファイルを取得
    try:
        image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"エラー: 入力フォルダ '{INPUT_DIR}' に画像ファイルが見つかりません。")
            return
    except FileNotFoundError:
        print(f"エラー: 入力フォルダ '{INPUT_DIR}' が見つかりません。パスを確認してください。")
        return

    print(f"{len(image_files)} 件の画像を処理します...")

    # 各画像ファイルに対して処理を実行
    for filename in tqdm(image_files, desc="マスク生成中"):
        image_path = os.path.join(INPUT_DIR, filename)

        # 画像を読み込む
        img = cv2.imread(image_path)
        if img is None:
            print(f"警告: ファイル '{filename}' を読み込めませんでした。スキップします。")
            continue

        # モデルで予測を実行
        try:
            results = model.predict(source=image_path, verbose=False)
        except Exception as e:
            print(f"警告: ファイル '{filename}' の予測中にエラーが発生しました。: {e}")
            continue

        # 検出結果からマスクを生成
        if results[0].masks is not None:
            # 元画像と同じサイズの黒いキャンバスを作成
            combined_mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

            # 検出されたすべてのマスクを1枚に結合
            for mask_tensor in results[0].masks.data:
                # テンソルをNumPy配列に変換し、元の画像サイズにリサイズ
                mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
                mask_resized = cv2.resize(mask_np, (img.shape[1], img.shape[0]))
                
                # 1枚のマスクに合成（ピクセル値が0より大きい部分を白(255)にする）
                combined_mask[mask_resized > 0] = 255
            
            # マスク画像を保存
            output_path = os.path.join(OUTPUT_DIR, filename)
            cv2.imwrite(output_path, combined_mask)
        # else:
            # オブジェクトが検出されなかった場合は何もしない（黒い画像は作成しない）

    print("すべての処理が完了しました。")

if __name__ == '__main__':
    create_and_save_masks()

In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import torch

# --- 設定 ---
# 1. 学習済みモデルのパス
MODEL_PATH = '/home/YOLO/0728_segmet/datasets/train2/weights/best.pt' # 'segmet'は'segment'のtypoかもしれませんが、前回のエラーに合わせています

# 2. 入力画像のフォルダパス
INPUT_DIR = '/home/data/0707_sam/data/crop'

# 3. マスク画像の保存先フォルダパス
OUTPUT_DIR = '/home/data/0707_sam/data/ratina_mask' # 保存先フォルダ名を変更
# ---

def create_highest_confidence_mask():
    """
    画像内の物体を検出し、最も信頼度の高いもの1つのマスクのみを生成して保存する。
    """
    # YOLOモデルを読み込む
    try:
        model = YOLO(MODEL_PATH)
        print(f"モデル '{MODEL_PATH}' を正常に読み込みました。")
    except Exception as e:
        print(f"エラー: モデルの読み込みに失敗しました。パスを確認してください。: {e}")
        return

    # 保存先フォルダが存在しない場合は作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"マスクの保存先: '{OUTPUT_DIR}'")

    # 処理対象の画像ファイルを取得
    try:
        image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"エラー: 入力フォルダ '{INPUT_DIR}' に画像ファイルが見つかりません。")
            return
    except FileNotFoundError:
        print(f"エラー: 入力フォルダ '{INPUT_DIR}' が見つかりません。パスを確認してください。")
        return

    print(f"{len(image_files)} 件の画像を処理します...")

    # 各画像ファイルに対して処理を実行
    # tqdmの引数はプログレスバーに関するものだけにする
    for filename in tqdm(image_files, desc="マスク生成中"):
        image_path = os.path.join(INPUT_DIR, filename)
        img = cv2.imread(image_path)
        if img is None:
            print(f"警告: ファイル '{filename}' を読み込めませんでした。スキップします。")
            continue

        # モデルで予測を実行する際に retina_masks=True を指定する
        try:
            results = model.predict(source=image_path, verbose=False, retina_masks=True)
        except Exception as e:
            print(f"警告: ファイル '{filename}' の予測中にエラーが発生しました。: {e}")
            continue

        # 検出結果があり、マスク情報も存在する場合のみ処理
        if results[0].boxes is not None and len(results[0].boxes) > 0 and results[0].masks is not None:
            # 信頼度が最も高い検出結果のインデックスを取得
            confidences = results[0].boxes.conf
            best_detection_index = torch.argmax(confidences)

            # 最も信頼度の高いマスクデータを取得
            best_mask_tensor = results[0].masks.data[best_detection_index]

            # マスク画像を生成
            mask_image = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
            mask_np = best_mask_tensor.cpu().numpy().astype(np.uint8)
            mask_resized = cv2.resize(mask_np, (img.shape[1], img.shape[0]))
            
            mask_image[mask_resized > 0] = 255
            
            # マスク画像を保存
            output_path = os.path.join(OUTPUT_DIR, filename)
            cv2.imwrite(output_path, mask_image)

    print("すべての処理が完了しました。")

if __name__ == '__main__':
    create_highest_confidence_mask()

画像内の複数しいたけをcrop，maskをとる

In [ ]:
# import torch
# from ultralytics import YOLO
# from PIL import Image
# import sys
# import os
# import cv2
# import numpy as np

# # -------------------------------------------------------------
# # 1. 画像ファイルのパスを指定 (★必ず変更してください)
# # -------------------------------------------------------------
# # ここに、処理したいお手元の画像ファイルの
# # 正確なパスを入力してください。
# LOCAL_IMAGE_PATH = '/home/data/0707_sam/data/IMG_2644.jpg' 

# # -------------------------------------------------------------
# # 2. 画像の読み込み
# # -------------------------------------------------------------
# # --- ここから下のコードは変更不要です ---
# print(f"ローカル画像を読み込んでいます: {LOCAL_IMAGE_PATH}")
# try:
#     # 指定されたパスから画像を読み込む
#     img = Image.open(LOCAL_IMAGE_PATH).convert("RGB")
# except FileNotFoundError:
#     # ファイルが見つからない場合のエラー処理
#     print(f"\n[エラー] 指定された画像ファイルが見つかりません。")
#     print(f"パスが正しいか確認してください: {LOCAL_IMAGE_PATH}")
#     sys.exit() # プログラムを終了
# except Exception as e:
#     # その他の画像読み込みエラー
#     print(f"\n[エラー] 画像の読み込み中に問題が発生しました。")
#     print(f"詳細: {e}")
#     sys.exit() # プログラムを終了

# # -------------------------------------------------------------
# # 3. モデルの読み込み & セグメンテーション実行
# # -------------------------------------------------------------
# print("\nセグメンテーションモデルを読み込んでいます...")
# model = YOLO('/home/YOLO/0708_maesyori/datasets/train/weights/best.pt') # セグメンテーション用モデル
# print("モデルの読み込み完了。セグメンテーションを実行します...")

# # モデルに画像を入力して推論を実行 (高品質マスクを生成)
# results = model(img, retina_masks=True)
# # results = model(img)
# result = results[0] # 最初の結果を取得
# print("セグメンテーションが完了しました。")

# # -------------------------------------------------------------
# # 4. クロップ画像と、それに合わせたマスクを個別に保存 (★★★ 新機能 ★★★)
# # -------------------------------------------------------------
# print("\nクロップ画像と、その構図に合わせたマスクを個別に保存します...")

# # 保存先ディレクトリを定義
# CROP_DIR = '/home/data/0819/crop'
# MASK_DIR = '/home/data/0819/mask'
# os.makedirs(CROP_DIR, exist_ok=True) # フォルダがなければ作成
# os.makedirs(MASK_DIR, exist_ok=True) # フォルダがなければ作成
# base_filename = os.path.splitext(os.path.basename(LOCAL_IMAGE_PATH))[0]

# # セグメンテーション結果がある場合のみ処理
# if result.masks is not None and result.boxes is not None:
#     num_objects = len(result.masks.data)
#     print(f"{num_objects}個の物体を検出しました。")
#     original_img_np = np.array(img)

#     # 検出された物体を一つずつループして処理
#     for i in range(num_objects):
#         # i番目のバウンディングボックス座標を取得
#         bbox_tensor = result.boxes.xyxy[i]
#         x1, y1, x2, y2 = map(int, bbox_tensor)
        
#         # --- (A) 認識した物体をクロップして保存 ---
        
#         # 1. 元の画像からバウンディングボックスの領域を切り抜く
#         cropped_object_rgb = original_img_np[y1:y2, x1:x2]
#         # 2. OpenCVで保存するために色順をRGBからBGRに変換
#         cropped_object_bgr = cv2.cvtColor(cropped_object_rgb, cv2.COLOR_RGB2BGR)
#         # 3. クロップした物体画像をファイルに保存
#         crop_filename = f"{base_filename}_object_{i}.png"
#         crop_path = os.path.join(CROP_DIR, crop_filename)
#         cv2.imwrite(crop_path, cropped_object_bgr)

#         # --- (B) クロップ画像に合わせたマスクを生成して保存 ---
        
#         # 1. まず、元画像と同じサイズのフルサイズのマスクを作成
#         full_mask_canvas = np.zeros(original_img_np.shape[:2], dtype=np.uint8)
#         mask_tensor = result.masks.data[i]
#         mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
#         mask_resized = cv2.resize(mask_np, (original_img_np.shape[1], original_img_np.shape[0]))
#         full_mask_canvas[mask_resized > 0] = 255
        
#         # 2. フルサイズのマスクを、(A)と同じ座標でクロップする
#         cropped_mask = full_mask_canvas[y1:y2, x1:x2]
        
#         # 3. クロップしたマスク画像をファイルに保存
#         mask_filename = f"{base_filename}_mask_{i}.png"
#         mask_path = os.path.join(MASK_DIR, mask_filename)
#         cv2.imwrite(mask_path, cropped_mask)

#     print(f"\n完了: {num_objects}個の物体とそのマスクをすべて保存しました。")
#     print(f"  - クロップ画像の保存先: ./{CROP_DIR}/")
#     print(f"  - マスク画像の保存先: ./{MASK_DIR}/")
# else:
#     print("物体は検出されませんでした。")

ローカル画像を読み込んでいます: /home/data/0707_sam/data/IMG_2644.jpg

セグメンテーションモデルを読み込んでいます...
モデルの読み込み完了。セグメンテーションを実行します...

0: 640x480 55 shiitakes, 28.7ms
Speed: 1.5ms preprocess, 28.7ms inference, 829.9ms postprocess per image at shape (1, 3, 640, 480)
セグメンテーションが完了しました。

クロップ画像と、その構図に合わせたマスクを個別に保存します...
55個の物体を検出しました。

完了: 55個の物体とそのマスクをすべて保存しました。
  - クロップ画像の保存先: .//home/data/0819/crop/
  - マスク画像の保存先: .//home/data/0819/mask/


In [3]:
import torch
from ultralytics import YOLO
from PIL import Image
import sys
import os
import cv2
import numpy as np
import glob

# -------------------------------------------------------------
# 1. 各種パスを指定 (★ここを変更してください)
# -------------------------------------------------------------
# 処理したい画像ファイルが入っているフォルダのパス
INPUT_IMAGE_DIR = '/home/data/0827_seido/org/C' 

# 学習済みモデルのパス
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'

# クロップ画像とマスクの保存先フォルダ
CROP_DIR = '/home/data/0827_seido/C/crop'
MASK_DIR = '/home/data/0827_seido/C/mask'

# -------------------------------------------------------------
# 2. モデルと保存先フォルダの準備
# -------------------------------------------------------------
# --- ここから下のコードは変更不要です ---
print("セグメンテーションモデルを読み込んでいます...")
try:
    model = YOLO(MODEL_PATH)
    print("モデルの読み込み完了。")
except Exception as e:
    print(f"\n[エラー] モデルの読み込みに失敗しました。")
    print(f"パスが正しいか確認してください: {MODEL_PATH}")
    print(f"詳細: {e}")
    sys.exit()

# 保存先ディレクトリを作成 (フォルダがなければ自動で作成)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
print(f"クロップ画像の保存先: {CROP_DIR}")
print(f"マスク画像の保存先:   {MASK_DIR}")

# -------------------------------------------------------------
# 3. 指定フォルダ内の画像を順番に処理
# -------------------------------------------------------------
print("\n画像処理を開始します...")

# 対象とする画像の拡張子
allowed_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']

# フォルダ内の画像ファイルリストを取得
image_paths = []
for ext in allowed_extensions:
    image_paths.extend(glob.glob(os.path.join(INPUT_IMAGE_DIR, ext)))

if not image_paths:
    print(f"\n[エラー] 指定されたフォルダに処理対象の画像が見つかりません。")
    print(f"パスを確認してください: {INPUT_IMAGE_DIR}")
    sys.exit()

print(f"{len(image_paths)} 件の画像ファイルを処理します。")

# 取得した画像パスでループ処理を開始
for i, image_path in enumerate(image_paths):
    print(f"\n--- [{i+1}/{len(image_paths)}] 処理中: {os.path.basename(image_path)} ---")
    
    # (A) 画像の読み込み
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"[警告] 画像の読み込みに失敗しました。このファイルをスキップします。詳細: {e}")
        continue # エラーが発生した場合は次の画像へ

    # (B) セグメンテーション実行
    results = model(img, retina_masks=True)
    result = results[0]

    # (C) クロップ画像とマスクを個別に保存
    base_filename = os.path.splitext(os.path.basename(image_path))[0]

    if result.masks is not None and result.boxes is not None:
        num_objects = len(result.masks.data)
        print(f"{num_objects}個の物体を検出しました。")
        original_img_np = np.array(img)

        # 検出された物体を一つずつループ
        for i in range(num_objects):
            # バウンディングボックス座標を取得
            bbox_tensor = result.boxes.xyxy[i]
            x1, y1, x2, y2 = map(int, bbox_tensor)
            
            # 1. 物体をクロップして保存
            cropped_object_rgb = original_img_np[y1:y2, x1:x2]
            cropped_object_bgr = cv2.cvtColor(cropped_object_rgb, cv2.COLOR_RGB2BGR)
            crop_filename = f"{base_filename}_object_{i}.png"
            crop_path = os.path.join(CROP_DIR, crop_filename)
            cv2.imwrite(crop_path, cropped_object_bgr)

            # 2. クロップ画像に合わせたマスクを生成して保存
            full_mask_canvas = np.zeros(original_img_np.shape[:2], dtype=np.uint8)
            mask_tensor = result.masks.data[i]
            mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
            mask_resized = cv2.resize(mask_np, (original_img_np.shape[1], original_img_np.shape[0]))
            full_mask_canvas[mask_resized > 0] = 255
            
            cropped_mask = full_mask_canvas[y1:y2, x1:x2]
            
            mask_filename = f"{base_filename}_mask_{i}.png"
            mask_path = os.path.join(MASK_DIR, mask_filename)
            cv2.imwrite(mask_path, cropped_mask)

        print(f"-> 保存完了: {os.path.basename(image_path)}")
    else:
        print("-> 物体は検出されませんでした。")

print("\n===================================")
print("すべての画像の処理が完了しました。")
print("===================================")

セグメンテーションモデルを読み込んでいます...
モデルの読み込み完了。
クロップ画像の保存先: /home/data/0827_seido/C/crop
マスク画像の保存先:   /home/data/0827_seido/C/mask

画像処理を開始します...
6 件の画像ファイルを処理します。

--- [1/6] 処理中: IMG_2659.jpg ---

0: 640x480 40 shiitakes, 25.7ms
Speed: 1.5ms preprocess, 25.7ms inference, 595.2ms postprocess per image at shape (1, 3, 640, 480)
40個の物体を検出しました。
-> 保存完了: IMG_2659.jpg

--- [2/6] 処理中: IMG_2658.jpg ---

0: 640x480 40 shiitakes, 15.7ms
Speed: 1.2ms preprocess, 15.7ms inference, 612.8ms postprocess per image at shape (1, 3, 640, 480)
40個の物体を検出しました。
-> 保存完了: IMG_2658.jpg

--- [3/6] 処理中: IMG_2660.jpg ---

0: 640x480 40 shiitakes, 15.4ms
Speed: 1.2ms preprocess, 15.4ms inference, 608.1ms postprocess per image at shape (1, 3, 640, 480)
40個の物体を検出しました。
-> 保存完了: IMG_2660.jpg

--- [4/6] 処理中: IMG_2657.jpg ---

0: 640x480 40 shiitakes, 15.4ms
Speed: 1.3ms preprocess, 15.4ms inference, 606.1ms postprocess per image at shape (1, 3, 640, 480)
40個の物体を検出しました。
-> 保存完了: IMG_2657.jpg

--- [5/6] 処理中: IMG_2656.jpg ---

0: 64

白い領域が複数のマスクを最も大きい領域を残して黒くする

In [8]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# --- 設定 ---
# 1. 処理したいマスク画像が入っているフォルダ
INPUT_DIR = '/home/data/0827_seido/C/mask'

# 2. 処理後のマスクを保存するフォルダ
OUTPUT_DIR = '/home/data/0827_seido/C/mask_to1'
# ---

def keep_largest_component(image: np.ndarray) -> np.ndarray:
    """
    二値画像を受け取り、最大の連結成分のみを残した画像を返す。
    """
    # 連結成分を解析
    # num_labels: 背景を含む連結成分の総数
    # labels: 各ピクセルがどの成分に属するかを示すラベル行列
    # stats: 各成分の統計情報（[x, y, width, height, area]）
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(image, connectivity=8)

    # 成分が1つ以下（背景のみ、または1つの塊のみ）の場合はそのまま返す
    if num_labels <= 2:
        return image

    # 背景（ラベル0）を除いた最大の成分を見つける
    # stats[1:, cv2.CC_STAT_AREA] で背景以外の全成分の面積を取得
    largest_component_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1

    # 新しいマスクを生成
    # 最大の成分に属するラベルを持つピクセルのみを白(255)にする
    cleaned_mask = np.zeros_like(image)
    cleaned_mask[labels == largest_component_label] = 255
    
    return cleaned_mask

def process_masks_in_directory():
    """
    指定されたディレクトリ内のすべてのマスク画像を処理する。
    """
    # 保存先フォルダが存在しない場合は作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"入力フォルダ: '{INPUT_DIR}'")
    print(f"出力フォルダ: '{OUTPUT_DIR}'")

    # 処理対象の画像ファイルを取得
    try:
        image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"エラー: 入力フォルダ '{INPUT_DIR}' に画像ファイルが見つかりません。")
            return
    except FileNotFoundError:
        print(f"エラー: 入力フォルダ '{INPUT_DIR}' が見つかりません。パスを確認してください。")
        return
        
    print(f"{len(image_files)} 件のマスク画像を処理します...")

    for filename in tqdm(image_files, desc="マスクをクリーンアップ中"):
        # 画像をグレースケールで読み込み
        image_path = os.path.join(INPUT_DIR, filename)
        mask_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if mask_image is None:
            continue

        # 最大の連結成分のみを残す処理
        cleaned_image = keep_largest_component(mask_image)

        # 結果を保存
        output_path = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(output_path, cleaned_image)

    print("すべての処理が完了しました。")


if __name__ == '__main__':
    process_masks_in_directory()

入力フォルダ: '/home/data/0827_seido/C/mask'
出力フォルダ: '/home/data/0827_seido/C/mask_to1'
240 件のマスク画像を処理します...


マスクをクリーンアップ中: 100%|██████████| 240/240 [00:00<00:00, 924.20it/s]

すべての処理が完了しました。


小さい画像（誤認識）を除外する
最も差の大きい間の中点がしきい値

In [5]:
# import os
# import cv2
# import matplotlib.pyplot as plt
# import numpy as np

# # --- 設定 ---
# # サイズを調べたい画像が入っているフォルダ
# INPUT_DIR = '/home/data/0827_seido/B/mask_to1'  # 調査するフォルダ名に合わせて変更してください
# # ---

# def analyze_image_sizes():
#     """フォルダ内の画像の面積を計算し、ヒストグラムで表示する。"""
#     image_areas = []
#     print(f"'{INPUT_DIR}'内の画像を調査しています...")
    
#     for filename in os.listdir(INPUT_DIR):
#         if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
#             continue
            
#         image_path = os.path.join(INPUT_DIR, filename)
#         img = cv2.imread(image_path)
#         if img is not None:
#             height, width, _ = img.shape
#             image_areas.append(height * width)

#     if not image_areas:
#         print("画像が見つかりませんでした。")
#         return

#     print(f"調査完了。{len(image_areas)}個の画像の面積データをプロットします。")
    
#     # ヒストグラムを作成
#     plt.figure(figsize=(10, 6))
#     plt.hist(image_areas, bins=50, edgecolor='black')
#     plt.title('Image Size Distribution (Pixel Area)')
#     plt.xlabel('Pixel Area (Width * Height)')
#     plt.ylabel('Number of Images')
#     plt.grid(True)
#     plt.show()

# if __name__ == '__main__':
#     analyze_image_sizes()

In [9]:
#50000ピクセル以下を別フォルダに，50000以下のcrop削除

import os
import cv2
import shutil
import numpy as np
from tqdm import tqdm

# --- 設定 ---
# 振り分けたいマスク画像が入っている元のフォルダ
SOURCE_DIR = '/home/data/0827_seido/A/mask_to1'

# 条件を満たした画像（大きい画像）の移動先フォルダ
DEST_DIR = '/home/data/0827_seido/A/mask_large'

# 条件から外れた画像（小さい画像）の移動先フォルダ
SMALL_DIR = '/home/data/0827_seido/A/mask_small'

# ★★★ 追加 ★★★
# 削除対象のクロップ画像が入っているフォルダ
CROP_DIR = '/home/data/0827_seido/A/crop'
# ★★★★★★★★

# 振り分けの基準となる面積のしきい値 (ピクセル単位)
AREA_THRESHOLD = 50000
# ---

def filter_and_cleanup_images():
    """
    固定のしきい値で画像を振り分け、小さい画像に対応するクロップ画像を削除する。
    """
    os.makedirs(DEST_DIR, exist_ok=True)
    os.makedirs(SMALL_DIR, exist_ok=True)

    try:
        filenames = [
            f for f in os.listdir(SOURCE_DIR) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))
        ]
    except FileNotFoundError:
        print(f"[エラー] 元フォルダが見つかりません: {SOURCE_DIR}")
        return

    if not filenames:
        print(f"'{SOURCE_DIR}' 内に処理対象の画像が見つかりませんでした。")
        return

    print(f"しきい値 {AREA_THRESHOLD} ピクセルに基づき、画像を振り分け、不要なクロップ画像を削除します...")
    
    deleted_count = 0
    for filename in tqdm(filenames, desc="画像を振り分け中"):
        source_path = os.path.join(SOURCE_DIR, filename)
        
        try:
            img = cv2.imread(source_path)
            if img is None:
                print(f"\n[警告] {filename} が読み込めませんでした。スキップします。")
                continue
                
            height, width, _ = img.shape
            area = height * width
            
            if area < AREA_THRESHOLD:
                # 小さい画像をSMALL_DIRへ移動
                shutil.copy(source_path, os.path.join(SMALL_DIR, filename))

                # ★★★ 追加: 対応するクロップ画像を削除する処理 ★★★
                # 1. マスクファイル名からクロップファイル名を生成
                #    (例: '..._mask_0.png' -> '..._object_0.png')
                crop_filename = filename.replace('_mask_', '_object_')
                
                # 2. 削除するクロップ画像のフルパスを作成
                crop_path_to_delete = os.path.join(CROP_DIR, crop_filename)
                
                # 3. ファイルが存在すれば削除
                if os.path.exists(crop_path_to_delete):
                    os.remove(crop_path_to_delete)
                    deleted_count += 1
                # ★★★★★★★★★★★★★★★★★★★★★★★★★★

            else:
                # 大きい画像をDEST_DIRへ移動
                shutil.copy(source_path, os.path.join(DEST_DIR, filename))

        except Exception as e:
            print(f"\n[エラー] {filename} の処理中に問題が発生しました: {e}")
            
    print("\n振り分けが完了しました。")
    print(f"  - 大きい画像の保存先: '{DEST_DIR}/'")
    print(f"  - 小さい画像の保存先: '{SMALL_DIR}/'")
    if deleted_count > 0:
        print(f"  - 削除されたクロップ画像: {deleted_count}枚")


if __name__ == '__main__':
    filter_and_cleanup_images()

しきい値 50000 ピクセルに基づき、画像を振り分け、不要なクロップ画像を削除します...


画像を振り分け中: 100%|██████████| 500/500 [00:00<00:00, 2198.10it/s]


振り分けが完了しました。
  - 大きい画像の保存先: '/home/data/0827_seido/A/mask_large/'
  - 小さい画像の保存先: '/home/data/0827_seido/A/mask_small/'
  - 削除されたクロップ画像: 4枚


In [ ]:
#動的な閾値決め，大きい方で分けられたり，ミスがないけど分けたりしちゃう．要修正

# import os
# import cv2
# import shutil
# import numpy as np
# from tqdm import tqdm

# # --- 設定 ---
# # 振り分けたい画像が入っている元のフォルダ
# SOURCE_DIR = '/home/data/0827_seido/C/mask_to1'  # 対象フォルダ名に合わせて変更してください

# # 条件を満たした画像（大きい画像）の移動先フォルダ
# DEST_DIR = '/home/data/0827_seido/C/mask_large'

# # 条件から外れた画像（小さい画像）の移動先フォルダ
# SMALL_DIR = '/home/data/0827_seido/C/mask_small'
# # ---

# def find_dynamic_threshold(areas: list) -> float:
#     """
#     画像の面積リストから、最大のギャップを見つけて動的なしきい値を返す。
#     """
#     # 面積が2つ未満の場合は処理しない
#     if len(areas) < 2:
#         return 0

#     # 面積を小さい順にソート
#     sorted_areas = np.sort(areas)
    
#     # 隣り合う面積の差分を計算
#     diffs = np.diff(sorted_areas)
    
#     # 最大の差分がある場所（インデックス）を見つける
#     max_gap_index = np.argmax(diffs)
    
#     # ギャップを形成する2つの値を取得
#     value_before_gap = sorted_areas[max_gap_index]
#     value_after_gap = sorted_areas[max_gap_index + 1]
    
#     # ギャップの中間値をしきい値として返す
#     threshold = (value_before_gap + value_after_gap) / 2
#     return threshold

# def dynamic_filter_images():
#     """
#     動的に決定したしきい値で画像を振り分ける。
#     """
#     # 1. まず全画像の面積をリストアップ
#     image_areas = []
#     filenames_map = {} # 面積とファイル名のマッピング
    
#     print(f"'{SOURCE_DIR}'内の画像を調査しています...")
#     for filename in os.listdir(SOURCE_DIR):
#         if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
#             continue
            
#         image_path = os.path.join(SOURCE_DIR, filename)
#         img = cv2.imread(image_path)
#         if img is not None:
#             height, width, _ = img.shape
#             area = height * width
#             image_areas.append(area)
#             filenames_map[filename] = area

#     if not image_areas:
#         print("画像が見つかりませんでした。")
#         return

#     # 2. 面積リストから動的にしきい値を決定
#     area_threshold = find_dynamic_threshold(image_areas)
#     print(f"\n分析の結果、動的な分離しきい値は {area_threshold:.2f} ピクセルに決定されました。")

#     # 3. 決定したしきい値で画像を振り分け
#     os.makedirs(DEST_DIR, exist_ok=True)
#     os.makedirs(SMALL_DIR, exist_ok=True)
    
#     print("決定したしきい値に基づき、画像を振り分けます...")
#     for filename, area in tqdm(filenames_map.items(), desc="画像を振り分け中"):
#         source_path = os.path.join(SOURCE_DIR, filename)
        
#         if area < area_threshold:
#             shutil.move(source_path, os.path.join(SMALL_DIR, filename))
#         else:
#             shutil.move(source_path, os.path.join(DEST_DIR, filename))
            
#     print("\n振り分けが完了しました。")
#     print(f"  - 大きい画像の保存先: './{DEST_DIR}/'")
#     print(f"  - 小さい画像の保存先: './{SMALL_DIR}/'")


# if __name__ == '__main__':
#     dynamic_filter_images()

'/home/data/0827_seido/C/mask_to1'内の画像を調査しています...

分析の結果、動的な分離しきい値は 332315.50 ピクセルに決定されました。
決定したしきい値に基づき、画像を振り分けます...


画像を振り分け中: 100%|██████████| 240/240 [00:00<00:00, 98515.65it/s]


振り分けが完了しました。
  - 大きい画像の保存先: './/home/data/0827_seido/C/mask_large/'
  - 小さい画像の保存先: './/home/data/0827_seido/C/mask_small/'


マスクの穴（白領域に囲まれた黒領域）を埋める

In [19]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# --- 設定 ---
# 1. 穴埋め処理をしたいマスク画像が入っているフォルダ
INPUT_DIR = '/home/data/0827_seido/A/mask_large'  # 対象フォルダ名に合わせて変更してください

# 2. 処理後のマスクを保存するフォルダ
OUTPUT_DIR = '/home/data/0827_seido/mask/A'
# ---

def fill_true_holes(image: np.ndarray) -> np.ndarray:
    """
    画像内の「完全に白で囲まれた黒い領域（真の穴）」のみを塗りつぶす。
    """
    # 念のため、画像を完全な二値（0と255）に変換
    _, binary_image = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    
    # 変更を加えるための画像のコピーを作成
    output_image = binary_image.copy()
    
    # 画像を反転させ、黒い領域を白い塊として扱う
    inverted_image = cv2.bitwise_not(binary_image)
    
    # 反転させた画像から、各黒領域の輪郭を見つける
    # cv2.RETR_EXTERNAL を使うことで、個々の塊（背景、穴）を別々に取得できる
    contours, _ = cv2.findContours(inverted_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 画像の高さと幅を取得
    h, w = binary_image.shape[:2]
    
    # 各輪郭（黒い塊）をチェック
    for cnt in contours:
        # 輪郭のバウンディングボックス（輪郭を囲む最小の長方形）を取得
        x, y, rect_w, rect_h = cv2.boundingRect(cnt)
        
        # バウンディングボックスが画像の端に接しているか判定
        is_on_border = (x == 0) or (y == 0) or (x + rect_w == w) or (y + rect_h == h)
        
        # もし画像の端に接していなければ、それは「真の穴」である
        if not is_on_border:
            # その穴を白(255)で塗りつぶす
            cv2.drawContours(output_image, [cnt], -1, color=255, thickness=cv2.FILLED)
            
    return output_image

def process_masks_in_directory():
    """
    指定されたディレクトリ内のすべてのマスク画像の穴を埋める。
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"入力フォルダ: '{INPUT_DIR}'")
    print(f"出力フォルダ: '{OUTPUT_DIR}'")

    try:
        image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"エラー: 入力フォルダ '{INPUT_DIR}' に画像ファイルが見つかりません。")
            return
    except FileNotFoundError:
        print(f"エラー: 入力フォルダ '{INPUT_DIR}' が見つかりません。パスを確認してください。")
        return
        
    print(f"{len(image_files)} 件のマスク画像を処理します...")

    for filename in tqdm(image_files, desc="マスクの真の穴を充填中"):
        image_path = os.path.join(INPUT_DIR, filename)
        mask_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if mask_image is None:
            continue

        # 真の穴を埋める処理
        filled_image = fill_true_holes(mask_image)

        # 結果を保存
        output_path = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(output_path, filled_image)

    print("すべての処理が完了しました。")

if __name__ == '__main__':
    process_masks_in_directory()

入力フォルダ: '/home/data/0827_seido/A/mask_large'
出力フォルダ: '/home/data/0827_seido/mask/A'
496 件のマスク画像を処理します...


マスクの真の穴を充填中: 100%|██████████| 496/496 [00:00<00:00, 1677.20it/s]

すべての処理が完了しました。


In [ ]:
# #最も小さい白ピクセル数の画像を見つける

# import os
# import cv2
# import numpy as np
# # Jupyter Notebook用のtqdmをインポート
# from tqdm.notebook import tqdm

# # --- 設定 ---
# # 画像が保存されているフォルダのパスを指定してください
# TARGET_DIR = '/home/data/0827_seido/C/mask_large'
# # ---

# def find_image_with_least_white_pixels():
#     """
#     指定されたフォルダ内の画像で、白ピクセルの数が最も少ないものを見つける。
#     """
#     min_white_pixels = float('inf')
#     min_white_pixel_image = None

#     try:
#         filenames = [
#             f for f in os.listdir(TARGET_DIR)
#             if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))
#         ]
#     except FileNotFoundError:
#         print(f"❌ [エラー] 指定されたフォルダが見つかりません: {TARGET_DIR}")
#         return

#     if not filenames:
#         print(f"📂 '{TARGET_DIR}' 内に処理対象の画像が見つかりませんでした。")
#         return

#     print(f"🖼️  {len(filenames)}枚の画像を分析します...")

#     for filename in tqdm(filenames, desc="画像を分析中"):
#         image_path = os.path.join(TARGET_DIR, filename)
#         try:
#             img = cv2.imread(image_path)
#             if img is None:
#                 print(f"\n[警告] {filename} が読み込めませんでした。スキップします。")
#                 continue

#             white_pixels = np.sum(np.all(img == [255, 255, 255], axis=2))

#             if white_pixels < min_white_pixels:
#                 min_white_pixels = white_pixels
#                 min_white_pixel_image = filename
#         except Exception as e:
#             print(f"\n[エラー] {filename} の処理中に問題が発生しました: {e}")

#     # --- 結果の表示 ---
#     if min_white_pixel_image is not None:
#         print("\n--- 分析完了 ---")
#         print(f"🏆 最も白ピクセルが少ない画像:")
#         print(f"   ファイル名: {min_white_pixel_image}")
#         print(f"   白ピクセル数: {min_white_pixels}個")
#     else:
#         # ★★★ ここを修正しました ★★★
#         # 分析対象が見つからなかった場合にメッセージを表示するように変更
#         print("\n分析対象となる有効な画像がありませんでした。")

# # --- 関数の実行 ---
# # Jupyter Notebookのセルで関数を直接呼び出します
# find_image_with_least_white_pixels()

🖼️  240枚の画像を分析します...


画像を分析中:   0%|          | 0/240 [00:00<?, ?it/s]


--- 分析完了 ---
🏆 最も白ピクセルが少ない画像:
   ファイル名: IMG_2658_mask_32.png
   白ピクセル数: 110574個


In [20]:
import os
import cv2
import numpy as np
from tqdm.notebook import tqdm

# --- 設定 ---
# 1. 元となるクロップ画像が入っているフォルダ
CROP_IMAGE_DIR = '/home/data/0827_seido/crop/A'

# 2. マスク画像が入っているフォルダ
MASK_IMAGE_DIR = '/home/data/0827_seido/mask/A'

# 3. マスク適用後の画像の保存先フォルダ
OUTPUT_DIR = '/home/data/0827_seido/combined/A'
# ---

def apply_masks_to_images():
    """
    クロップ画像フォルダとマスク画像フォルダから対応するファイルを読み込み、
    マスクを適用した画像を生成して保存する。
    """
    # 保存先フォルダがなければ作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"入力(クロップ画像): '{CROP_IMAGE_DIR}'")
    print(f"入力(マスク画像)  : '{MASK_IMAGE_DIR}'")
    print(f"出力先            : '{OUTPUT_DIR}'")

    try:
        # クロップ画像フォルダ内の画像ファイルリストを取得
        crop_files = [
            f for f in os.listdir(CROP_IMAGE_DIR) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        print(f"❌ [エラー] クロップ画像フォルダが見つかりません: {CROP_IMAGE_DIR}")
        return

    if not crop_files:
        print(f"📂 '{CROP_IMAGE_DIR}' 内に処理対象の画像が見つかりませんでした。")
        return

    print(f"\n🖼️  {len(crop_files)}枚の画像にマスクを適用します...")
    
    success_count = 0
    not_found_count = 0
    for crop_filename in tqdm(crop_files, desc="マスク適用中"):
        # 対応するマスクファイル名を生成
        # (例: '..._object_0.png' -> '..._mask_0.png')
        mask_filename = crop_filename.replace('_object_', '_mask_')
        
        # 各ファイルのフルパスを作成
        crop_path = os.path.join(CROP_IMAGE_DIR, crop_filename)
        mask_path = os.path.join(MASK_IMAGE_DIR, mask_filename)
        
        # 対応するマスクファイルが存在するかチェック
        if not os.path.exists(mask_path):
            # print(f"\n[警告] {crop_filename} に対応するマスクが見つかりません。スキップします。")
            not_found_count += 1
            continue
            
        try:
            # 画像とマスクを読み込む
            crop_image = cv2.imread(crop_path)
            # マスクはグレースケールで読み込む
            mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if crop_image is None or mask_image is None:
                continue

            # マスクを二値化（白か黒かにする）
            _, binary_mask = cv2.threshold(mask_image, 127, 255, cv2.THRESH_BINARY)
            
            # マスクを3チャンネルのカラー画像に変換して、元の画像とサイズを合わせる
            mask_3ch = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)

            # マスクが白い部分だけ元の画像を残し、黒い部分は黒にする
            # (bitwise_andは両方の画像でピクセル値が0でない部分だけを残す)
            masked_image = cv2.bitwise_and(crop_image, mask_3ch)

            # 結果を保存
            output_path = os.path.join(OUTPUT_DIR, crop_filename)
            cv2.imwrite(output_path, masked_image)
            success_count += 1
            
        except Exception as e:
            print(f"\n[エラー] {crop_filename} の処理中に問題が発生しました: {e}")

    print("\n--- 処理完了 ---")
    print(f"✅ 正常に処理された画像: {success_count}枚")
    if not_found_count > 0:
        print(f"⚠️  対応するマスクが見つからなかった画像: {not_found_count}枚")

# --- 関数の実行 ---
apply_masks_to_images()

入力(クロップ画像): '/home/data/0827_seido/crop/A'
入力(マスク画像)  : '/home/data/0827_seido/mask/A'
出力先            : '/home/data/0827_seido/combined/A'

🖼️  496枚の画像にマスクを適用します...


マスク適用中:   0%|          | 0/496 [00:00<?, ?it/s]


--- 処理完了 ---
✅ 正常に処理された画像: 496枚


In [23]:
#IMG_2648_mask_0.png→IMG_6480_mask.png(crop，combinedも同様)

import os
from tqdm.notebook import tqdm

# --- 設定 ---
# 1. crop画像フォルダ
CROP_DIR = '/home/data/0827_seido/crop/C'

# 2. mask画像フォルダ
MASK_DIR = '/home/data/0827_seido/mask/C'

# 3. マスク適用済み画像（combined）フォルダ
COMBINED_DIR = '/home/data/0827_seido/combined/C'
# ---

def rename_files_in_folder(target_dir: str, suffix: str):
    """
    指定されたフォルダ内の画像ファイル名を新しいルールに従って一括変更する。
    """
    print(f"--- フォルダ '{os.path.basename(target_dir)}' の処理を開始 ---")
    
    try:
        # フォルダ内のファイルリストを取得
        filenames = [
            f for f in os.listdir(target_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
    except FileNotFoundError:
        print(f"❌ [エラー] フォルダが見つかりません: {target_dir}")
        return

    if not filenames:
        print("📂 処理対象の画像が見つかりませんでした。")
        return
        
    renamed_count = 0
    # プログレスバーを表示しながら処理
    for filename in tqdm(filenames, desc=f"{suffix} 画像をリネーム中"):
        try:
            # 拡張子とそれ以外（基底名）に分離 -> ('IMG_2642_object_0', '.png')
            base, ext = os.path.splitext(filename)
            
            # '_'で分割 -> ['IMG', '2642', 'object', '0']
            parts = base.split('_')
            
            # 元画像の番号（'2642'）と末尾の番号（'0'）を取得
            original_num_str = parts[1]
            index_str = parts[-1] # 末尾の要素を取得
            
            # 新しい名前の要素を組み立てる
            xxx = original_num_str[-3:] # 下3桁を取得 -> '642'
            y = index_str               # -> '0'
            
            # 新しいファイル名を生成
            new_filename = f"IMG_{xxx}{y}_{suffix}{ext}" # -> 'IMG_6420_crop.png'
            
            # 古いパスと新しいパスを定義
            old_path = os.path.join(target_dir, filename)
            new_path = os.path.join(target_dir, new_filename)
            
            # ファイル名を変更
            if old_path != new_path:
                os.rename(old_path, new_path)
                renamed_count += 1

        except IndexError:
            # ファイル名が 'IMG_xxxx_suffix_y' の形式でない場合はスキップ
            print(f"\n[警告] '{filename}' は期待される形式でないため、スキップしました。")
        except Exception as e:
            print(f"\n[エラー] '{filename}' の処理中に問題が発生しました: {e}")

    print(f"✅ {renamed_count}件のファイル名を変更しました。\n")

# --- メインの処理 ---
if __name__ == '__main__':
    # 各フォルダに対してリネーム処理を実行
    rename_files_in_folder(CROP_DIR, 'crop')
    rename_files_in_folder(MASK_DIR, 'mask')
    rename_files_in_folder(COMBINED_DIR, 'combined')
    
    print("すべてのリネーム処理が完了しました。")

--- フォルダ 'C' の処理を開始 ---


crop 画像をリネーム中:   0%|          | 0/240 [00:00<?, ?it/s]

✅ 240件のファイル名を変更しました。

--- フォルダ 'C' の処理を開始 ---


mask 画像をリネーム中:   0%|          | 0/240 [00:00<?, ?it/s]

✅ 240件のファイル名を変更しました。

--- フォルダ 'C' の処理を開始 ---


combined 画像をリネーム中:   0%|          | 0/240 [00:00<?, ?it/s]

✅ 240件のファイル名を変更しました。

すべてのリネーム処理が完了しました。
